# Lecture 7: Goodness of fit, model comparison, and hypothesis testing

**Unit 2, September 17, 2026**

We continue the **same quantum-optics damped oscillation example from Lecture 6**,

$$
P(t)=P_0 + C e^{-t/T_2}\cos(2\pi f t+\phi),
$$

so that today's new material is statistical rather than physical. We will stay in the **weighted least-squares** framework and ask two different questions:

1. **Goodness of fit:** Is one model compatible with the data and stated uncertainties?
2. **Model comparison:** Does a more flexible model improve the fit enough to justify its additional parameter(s)?

Likelihood fitting will be treated in Lecture 8. Today we will only note the Gaussian connection when it helps explain the $\Delta\chi^2$ test.


## Learning goals

By the end of this lecture you should be able to:

- perform and interpret a weighted nonlinear least-squares fit,
- compute $\chi^2$, the number of degrees of freedom, reduced $\chi^2$, and a goodness-of-fit p-value,
- explain why goodness of fit and model comparison answer different questions,
- use $\Delta\chi^2$ for an appropriate nested-model hypothesis test,
- recognize situations where the usual $\chi^2$ approximations may be unreliable,
- support a model-choice statement with both numerical diagnostics and residual plots.


# Part 1: Shared data and model from Lecture 6

The model is

$$
P(t;\boldsymbol{\theta}) = P_0 + C e^{-t/T_2}\cos(2\pi f t + \phi).
$$

As in Lecture 6, we will use a short observation window. This is useful pedagogically because the oscillation frequency and phase are visible, while the much longer coherence time $T_2$ is only weakly constrained.

The measured probabilities come from repeated binary shots. For today's weighted least-squares treatment, we summarize each time point by the observed probability and an estimated standard uncertainty $\sigma_i$.


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

rng = np.random.default_rng(20260917)
print(f"NumPy {np.__version__} | pandas {pd.__version__}")


In [ ]:
def damped_oscillation_probability(time_us, offset, contrast, frequency_MHz, phase, coherence_time_us):
    """Damped oscillation model for a measured excited-state probability."""
    envelope = np.exp(-time_us / coherence_time_us)
    angle = 2.0 * np.pi * frequency_MHz * time_us + phase
    return offset + contrast * envelope * np.cos(angle)

parameter_names = ["offset", "contrast", "frequency_MHz", "phase", "coherence_time_us"]

true_parameters = {
    "offset": 0.50,
    "contrast": 0.22,
    "frequency_MHz": 0.82,
    "phase": 0.45,
    "coherence_time_us": 8.0,
}

initial_guess = [0.50, 0.20, 0.80, 0.30, 10.0]
parameter_bounds = (
    [0.0, 0.0, 0.1, -np.pi, 0.2],
    [1.0, 0.5, 2.0,  np.pi, 50.0],
)

quantum_data = pd.DataFrame({"time_us": np.linspace(0.0, 1.8, 24)})
quantum_data["true_probability"] = damped_oscillation_probability(
    quantum_data["time_us"],
    *true_parameters.values(),
).clip(0.02, 0.98)

quantum_data["shots"] = 180
quantum_data["excited_counts"] = rng.binomial(
    quantum_data["shots"],
    quantum_data["true_probability"],
)
quantum_data["probability"] = quantum_data["excited_counts"] / quantum_data["shots"]

# Approximate standard uncertainty for a measured binomial proportion.
# The small floor prevents a zero uncertainty in extreme samples.
quantum_data["sigma_probability"] = np.sqrt(
    np.maximum(
        quantum_data["probability"] * (1.0 - quantum_data["probability"]) / quantum_data["shots"],
        1.0 / quantum_data["shots"]**2,
    )
)

smooth_quantum_data = pd.DataFrame({"time_us": np.linspace(0.0, 1.8, 500)})
quantum_data.head()


In [ ]:
def normalized_residuals(parameters, data, model_function):
    prediction = model_function(data["time_us"], *parameters)
    return (data["probability"] - prediction) / data["sigma_probability"]


def fit_model(label, model_function, parameter_names, initial_guess, bounds):
    popt, pcov = curve_fit(
        model_function,
        quantum_data["time_us"],
        quantum_data["probability"],
        sigma=quantum_data["sigma_probability"],
        p0=initial_guess,
        bounds=bounds,
        absolute_sigma=True,
        maxfev=20000,
    )

    residuals = normalized_residuals(popt, quantum_data, model_function)
    chi2_value = np.sum(residuals**2)
    n_parameters = len(popt)
    ndf = len(quantum_data) - n_parameters

    return {
        "label": label,
        "model_function": model_function,
        "parameter_names": parameter_names,
        "popt": popt,
        "pcov": pcov,
        "n_parameters": n_parameters,
        "chi2": chi2_value,
        "ndf": ndf,
        "reduced_chi2": chi2_value / ndf,
        "p_value": chi2.sf(chi2_value, ndf),
        "residuals": residuals,
    }


full_fit = fit_model(
    "full damped model",
    damped_oscillation_probability,
    parameter_names,
    initial_guess,
    parameter_bounds,
)

full_fit_summary = pd.DataFrame({
    "parameter": parameter_names,
    "estimate": full_fit["popt"],
    "uncertainty": np.sqrt(np.diag(full_fit["pcov"])),
    "true_value": [true_parameters[name] for name in parameter_names],
})
full_fit_summary

fig, ax = plt.subplots()
ax.errorbar(
    quantum_data["time_us"],
    quantum_data["probability"],
    yerr=quantum_data["sigma_probability"],
    fmt="o",
    capsize=3,
    label="measured probability",
)
ax.plot(
    smooth_quantum_data["time_us"],
    damped_oscillation_probability(
        smooth_quantum_data["time_us"],
        *full_fit["popt"],
    ),
    label="fitted model",
)
ax.set_xlabel("time (microseconds)")
ax.set_ylabel("excited-state probability")
ax.set_title("Lecture 6 damped-oscillation example")
ax.set_ylim(0.0, 1.0)
ax.legend()
fig.tight_layout()


### Discussion question 1

Suppose the error bars on every point were accidentally made **twice too large**, but the fitted curve hardly changed. What would happen qualitatively to $\chi^2$, reduced $\chi^2$, and the goodness-of-fit p-value?

Keep this in mind: a goodness-of-fit test evaluates **the model together with the uncertainty model**.


# Part 2: From weighted residuals to $\chi^2$

The statistic $\chi^2$ is not just a convenient formula. It comes from a **generative model** for the measurement process.

Suppose the experiment reports a measured value $y_i$ at a known time $t_i$, and suppose the model prediction for parameters $\boldsymbol{\theta}$ is

$$
m_i(\boldsymbol{\theta}) = P(t_i;\boldsymbol{\theta}).
$$

The weighted least-squares model assumes that the measured value was generated as

$$
y_i = m_i(\boldsymbol{\theta}) + \epsilon_i,
$$

with independent Gaussian measurement errors

$$
\epsilon_i \sim \mathcal{N}(0,\sigma_i^2).
$$

Here $\mathcal{N}(\mu,\sigma^2)$ denotes a **normal**, or Gaussian, distribution with mean $\mu$ and variance $\sigma^2$. The script $\mathcal{N}$ is just conventional notation for this probability distribution.

Equivalently,

$$
y_i \sim \mathcal{N}\!\left(m_i(\boldsymbol{\theta}),\sigma_i^2\right).
$$

A **probability model** tells us the probability density for possible data values if the parameters are known. A **likelihood** uses the same probability model after the data have been observed, but treats the parameters as the unknown quantities. In symbols,

$$
L(\boldsymbol{\theta}) = p(\mathrm{data}\mid\boldsymbol{\theta}).
$$

For independent measurements, the joint probability density is the product of the individual probability densities. Under this Gaussian measurement model, the likelihood for the observed data is

$$
L(\boldsymbol{\theta})=
p(\{y_i\}_{i=1}^{N}\mid\boldsymbol{\theta})
=\prod_{i=1}^{N}
\frac{1}{\sqrt{2\pi\sigma_i^2}}
\exp\left[-\frac{1}{2}
\left(\frac{y_i-m_i(\boldsymbol{\theta})}{\sigma_i}\right)^2\right].
$$

Taking $-2\ln L$ gives

$$
-2\ln L(\boldsymbol{\theta}) =
\sum_{i=1}^{N}
\left[\frac{y_i-m_i(\boldsymbol{\theta})}{\sigma_i}\right]^2
+ \mathrm{constant},
$$

where the constant does not depend on the fit parameters if the $\sigma_i$ values are known. Therefore maximizing this Gaussian likelihood is equivalent to minimizing

$$
\chi^2(\boldsymbol{\theta})
= \sum_{i=1}^{N}
\left[\frac{y_i-P(t_i;\boldsymbol{\theta})}{\sigma_i}\right]^2.
$$

The normalized residual is

$$
r_i = \frac{y_i-m_i}{\sigma_i}.
$$

If the model is correct and the measurement errors really are independent Gaussian variables with the stated $\sigma_i$, then before fitting each normalized residual is approximately a standard normal random variable,

$$
r_i \sim \mathcal{N}(0,1).
$$

The sum of squares of independent standard-normal variables has a chi-square distribution. This is why $\chi^2$ can be used as a goodness-of-fit statistic.

After fitting $k$ parameters, the residuals are no longer all independent because the same data were used to estimate the parameters. For a well-behaved fit we use

$$
\nu = N-k
$$

as the number of **degrees of freedom** and compare the minimized $\chi^2$ with a $\chi^2_\nu$ reference distribution. For nonlinear models this interpretation is an approximation that is best when the model is locally close to linear near the optimum and parameters are identifiable.

This framing is important: if the data are not well described by independent Gaussian errors with known $\sigma_i$, then ordinary weighted least squares may still be useful as a diagnostic, but its p-values and parameter uncertainties no longer have the simple interpretation above.


## Reduced $\chi^2$ and the Pearson goodness-of-fit p-value

The **reduced chi-square** is

$$
\chi^2_\mathrm{red} = \frac{\chi^2}{\nu}.
$$

In this notebook we avoid writing the reduced chi-square as $\chi^2_\nu$, because $\chi^2_\nu$ can also mean a chi-square distribution with $\nu$ degrees of freedom.

For a correct model with correctly estimated uncertainties,

$$
E[\chi^2] \approx \nu,
\qquad
E[\chi^2/\nu] \approx 1.
$$

Here $E[Y]$ means the **expectation value**, or long-run average, of the random variable $Y$ over many repeated experiments generated from the assumed model. So $E[\chi^2]=\nu$ means that if we repeated the same experiment many times under the model assumptions, the average value of the $\chi^2$ statistic would approach the number of degrees of freedom.

But reduced $\chi^2\approx 1$ is only a rule of thumb. The expected fluctuations depend on $\nu$, so the more precise diagnostic is the **Pearson chi-square goodness-of-fit test**.

For the Gaussian residual model used here, the Pearson test statistic is the same minimized weighted sum of squares:

$$
X^2_\mathrm{obs}=\sum_i \left[\frac{y_i-m_i(\hat\theta)}{\sigma_i}\right]^2.
$$

The goodness-of-fit p-value is the upper-tail probability

$$
p_\mathrm{GOF}
= \Pr\left(X^2 \ge X^2_\mathrm{obs};\nu\right)
= \int_{X^2_\mathrm{obs}}^\infty f_{\chi^2_\nu}(x)\,dx.
$$

This goodness-of-fit (GOF) p-value is the probability of observing a $\chi^2$ value at least as large as the observed one, for a given model.  What does that mean in practice:
- A **small** p-value says that residuals this large would be unusual if the model and uncertainty assumptions were correct
- A **large** p-value does **not** mean that the model is probably true.


### How the chi-square reference distribution changes with degrees of freedom

The next plot shows several chi-square probability distributions, $f_{\chi^2_\nu}(x)$. Each curve has a different number of degrees of freedom $\nu$. The dashed vertical line marks the expectation value

$$
E[\chi^2]=\nu.
$$

As $\nu$ increases, the distribution moves to larger $\chi^2$ values and becomes relatively narrower compared with its mean. This is why a raw $\chi^2$ value is not meaningful without also reporting $\nu$.


In [ ]:
# Instructor demo: chi-square reference distributions for several degrees of freedom.
nu_values = [2, 5, 10, 20, 40, 100]
chi2_grid = np.linspace(0, 120, 800)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for nu_value in nu_values:
    distribution_values = chi2.pdf(chi2_grid, nu_value)
    ax.plot(chi2_grid, distribution_values, label=fr"$\nu={nu_value}$")
    ax.axvline(nu_value, color=ax.lines[-1].get_color(), ls="--", alpha=0.55)

ax.set_xlabel(r"$\chi^2$")
ax.set_ylabel("probability density")
ax.set_title(r"Chi-square reference distributions: $E[\chi^2]=\nu$")
ax.legend(title="degrees of freedom")
fig.tight_layout()


### In-class coding activity 1 — fit quality and residuals (12 minutes)

Complete the code below to calculate the goodness-of-fit quantities for the full damped model.

Then make a normalized-residual plot and answer:

- Is the overall size of the residuals plausible?
- Do the residuals show visible structure that a single number could hide?
- What would you report: $\chi^2$, reduced $\chi^2$, p-value, or all three?


In [ ]:
# TODO 1: Build a one-row goodness-of-fit table.
# Hints:
#   - full_fit already contains the fitted parameters and residuals.
#   - scipy.stats -> chi2.sf(observed_chi2, ndf) gives the upper-tail probability.

chi2_value = ...
ndf = ...
reduced_chi2 = ...
p_value = ...

full_model_quality = pd.DataFrame([
    {
        "model": "full damped model",
        "chi2": chi2_value,
        "ndf": ndf,
        "reduced_chi2": reduced_chi2,
        "p_value": p_value,
    }
])
full_model_quality


### Pearson chi-square test as an area under a reference distribution

The figure below shows the reference distribution used by the Pearson chi-square goodness-of-fit test. The curve is the chi-square probability density for $\nu=N-k$ degrees of freedom. The vertical dashed line is the observed test statistic $X^2_\mathrm{obs}$ from the fit.

The shaded area to the right is the p-value:

$$
p_\mathrm{GOF}=\Pr\left(X^2 \ge X^2_\mathrm{obs}\right).
$$

This is why the test is an **upper-tail** test: a poor fit produces residuals that are too large, which pushes the statistic to the right side of the reference distribution.

The statement “reduced $\chi^2$ should be near 1” can be misleading if we forget that $\chi^2$ fluctuates. The width of the reference distribution is

$$
\mathrm{Var}(\chi^2)=2\nu.
$$

So the same reduced-$\chi^2$ value can be more or less surprising depending on the number of degrees of freedom.


In [ ]:
# Instructor demo: visualize the Pearson chi-square goodness-of-fit test.
ndf_demo = full_fit["ndf"]
observed_demo = full_fit["chi2"]

x = np.linspace(0, max(2.2 * ndf_demo, 1.4 * observed_demo), 500)

fig, ax = plt.subplots()
ax.plot(x, chi2.pdf(x, ndf_demo), label=fr"$\chi^2_{{{ndf_demo}}}$ reference distribution")
ax.axvline(observed_demo, ls="--", label=fr"observed $X^2={observed_demo:.1f}$")
ax.fill_between(
    x,
    0,
    chi2.pdf(x, ndf_demo),
    where=(x >= observed_demo),
    alpha=0.25,
    label=fr"upper tail, $p={full_fit['p_value']:.3f}$",
)
ax.set_xlabel(r"$\chi^2$")
ax.set_ylabel("probability density")
ax.set_title("Pearson chi-square goodness-of-fit p-value")
ax.legend()
fig.tight_layout()


# Part 3: Goodness of fit is not model comparison

These are different statistical questions:

| Question | Typical statistic | Interpretation |
| --- | --- | --- |
| **Goodness of fit:** Is model $M$ compatible with the data and error bars? | $\chi^2$, $\nu$, $p_\mathrm{GOF}$ | Evaluates one model against its own expected residual fluctuations |
| **Model comparison:** Does $M_1$ improve enough over $M_0$ to justify extra flexibility? | $\Delta\chi^2$ for suitable nested models | Evaluates the improvement when constraints are relaxed |

Two important consequences:

1. Two models can both have acceptable goodness-of-fit p-values, while one is still unnecessarily complicated.
2. A more flexible model almost always lowers $\chi^2$; the decrease alone is not evidence that the extra parameter is needed.


### Discussion question 2

Suppose a 4-parameter model has $\chi^2=22$ and a 5-parameter model has $\chi^2=21$ on the same data set.

Is the 5-parameter model “better”? What additional information do you need before making a statistical statement?


# Part 4: Nested hypotheses and $\Delta\chi^2$

A model $M_0$ is **nested** inside a larger model $M_1$ if $M_0$ can be obtained by fixing one or more parameters of $M_1$ to specific values.

For example, the full damped model

$$
P(t)=P_0+C e^{-t/T_2}\cos(2\pi f t+\phi)
$$

contains the restricted hypothesis $\phi=0$ as a special case.

For nested weighted least-squares fits, define

$$
\Delta\chi^2
= \chi^2_\mathrm{restricted}
- \chi^2_\mathrm{larger}.
$$

If the restricted model is correct **and the usual regularity conditions are reasonable**, then asymptotically

$$
\Delta\chi^2 \sim \chi^2_{\Delta k},
$$

where $\Delta k$ is the number of additional free parameters in the larger model. Thus

$$
p_\Delta
= \Pr\left(\chi^2_{\Delta k}\ge\Delta\chi^2_\mathrm{obs}\right).
$$

This is the least-squares form of the same asymptotic result that underlies the likelihood-ratio test for Gaussian errors; we will discuss likelihood fits next lecture.


## When is the usual $\Delta\chi^2$ test appropriate?

Use the standard $\chi^2_{\Delta k}$ reference distribution when the following are reasonably true:

- the models are **nested**,
- both fits use the **same data and uncertainty model**,
- the restricted value is in the **interior** of the allowed parameter space rather than on a physical boundary,
- the extra parameter is **identifiable** under the larger model,
- both optimizations have actually found the relevant minima.

If these conditions fail, the numerical value of $\Delta\chi^2$ is still a useful description of improvement, but the simple $\chi^2_{\Delta k}$ p-value may not be calibrated correctly.


In [ ]:
# Restricted model for the main nested-model exercise: phi = 0.
def zero_phase_probability(time_us, offset, contrast, frequency_MHz, coherence_time_us):
    fixed_phase = 0.0
    return damped_oscillation_probability(
        time_us,
        offset,
        contrast,
        frequency_MHz,
        fixed_phase,
        coherence_time_us,
    )


def delta_chi2_test(restricted_fit, larger_fit):
    delta_chi2 = restricted_fit["chi2"] - larger_fit["chi2"]
    delta_parameters = larger_fit["n_parameters"] - restricted_fit["n_parameters"]
    return {
        "restricted_model": restricted_fit["label"],
        "larger_model": larger_fit["label"],
        "delta_chi2": delta_chi2,
        "delta_parameters": delta_parameters,
        "p_value": chi2.sf(delta_chi2, delta_parameters),
    }


### In-class coding activity 2 — test whether the phase is needed (12 minutes)

The full model allows $\phi$ to float. The restricted model fixes $\phi=0$.

1. Fit the restricted model using `fit_model`.
2. Compute $\Delta\chi^2$ and $\Delta k$.
3. Convert $\Delta\chi^2$ to a p-value.
4. Write one sentence that answers the scientific question: **Do these data require a nonzero phase?**

Do not decide by comparing reduced $\chi^2$ values alone.


In [ ]:
# TODO 1: Fit the phi = 0 restricted model.
#zero_phase_fit = ...

print(f"Full fit chi2: {full_fit['chi2']:.3f}, p-value: {full_fit['p_value']:.3f}")
print(f"Zero-phase fit chi2: {zero_phase_fit['chi2']:.3f}, p-value: {zero_phase_fit['p_value']:.3f}")

In [ ]:
# TODO 2: After zero_phase_fit is defined, compute the nested-model test.
# You may use delta_chi2_test(...) or calculate the pieces explicitly.

phase_test = pd.DataFrame([
    {
        "restricted_model": "phi = 0 model",
        "larger_model": "full damped model",
        "delta_chi2": ..., # Delta chi-square: restricted minus larger
        "delta_parameters": ..., # Delta k
        "p_value": ...,  # p-value for the nested-model test
    }
])
phase_test


# Part 5: Caveats that matter in real fits

### 1. Weakly identified parameters

Our data stop at $1.8\,\mu\mathrm{s}$, while the true coherence time is $T_2=8\,\mu\mathrm{s}$. The envelope therefore changes only modestly across the observed window. A optimizer may return a best-fit $T_2$, but that does not mean the data constrain it well.

A weakly identified parameter often shows up as a large uncertainty, strong covariance with other parameters, a shallow $\chi^2$ profile, sensitivity to bounds, or unstable fit results.

### 2. Parameters near physical bounds

If the null hypothesis puts a parameter on a boundary—e.g. testing a nonnegative signal amplitude with $H_0:C=0$—the standard Wilks/$\chi^2_{\Delta k}$ result can fail. The null distribution may no longer be an ordinary chi-square distribution.

### 3. Overfitting

Adding parameters cannot increase the minimum $\chi^2$ for truly nested models. Therefore “the more complex fit has a smaller $\chi^2$” is not, by itself, evidence in its favor.

### 4. Interpreting p-values

A p-value is computed **assuming the null model and its statistical assumptions are correct**. It is not

$$
\Pr(\text{model is true}\mid\text{data}).
$$

Very small p-values can reflect model misspecification, underestimated uncertainties, outliers, correlations, or a genuinely wrong physical hypothesis.


### 5. Statistical and systematic uncertainty

A fit covariance matrix usually describes **statistical uncertainty** under the assumed measurement model: finite sample size, counting fluctuations, shot noise, or repeated-measurement scatter. It does not automatically include **systematic uncertainty** from calibration, selection cuts, background shape, efficiency, resolution, or model choice.

A complete physics result should say which uncertainty sources were included in the fit and which were only checked qualitatively. In later project work, a common practical approach is to repeat the analysis under reasonable systematic variations and compare the shifts with the statistical uncertainty.

### 6. Coverage language

A confidence interval is a property of a procedure: over many repeated experiments generated under the assumed model, a nominal 68% interval should contain the true parameter about 68% of the time. That is a different statement from “there is a 68% probability that this specific fitted interval contains the true value.” The distinction matters when reporting frequentist intervals from $\chi^2$, covariance matrices, or likelihood-ratio tests.


### Discussion question 3

Imagine that freeing $T_2$ changes $\chi^2$ only slightly, but the fitted value of $T_2$ moves to the upper bound of the optimizer and its reported uncertainty becomes enormous.

What is the scientifically honest conclusion: “$T_2$ is very large,” “there is no damping,” or “this data set does not identify $T_2$ well”? Why?


In [ ]:
# A second restricted model: fix T2 to the independently calibrated value 8 microseconds.
def fixed_t2_probability(time_us, offset, contrast, frequency_MHz, phase):
    fixed_coherence_time_us = 8.0
    return damped_oscillation_probability(
        time_us,
        offset,
        contrast,
        frequency_MHz,
        phase,
        fixed_coherence_time_us,
    )

fixed_t2_fit = fit_model(
    "fixed T2 model",
    fixed_t2_probability,
    ["offset", "contrast", "frequency_MHz", "phase"],
    [0.50, 0.20, 0.80, 0.30],
    ([0.0, 0.0, 0.1, -np.pi], [1.0, 0.5, 2.0, np.pi]),
)

comparison_table = pd.DataFrame([
    {
        "model": fixed_t2_fit["label"],
        "n_parameters": fixed_t2_fit["n_parameters"],
        "chi2": fixed_t2_fit["chi2"],
        "ndf": fixed_t2_fit["ndf"],
        "reduced_chi2": fixed_t2_fit["reduced_chi2"],
        "gof_p_value": fixed_t2_fit["p_value"],
    },
    {
        "model": full_fit["label"],
        "n_parameters": full_fit["n_parameters"],
        "chi2": full_fit["chi2"],
        "ndf": full_fit["ndf"],
        "reduced_chi2": full_fit["reduced_chi2"],
        "gof_p_value": full_fit["p_value"],
    },
])
comparison_table


### In-class coding activity 3 — weak identification and scientific reporting (8 minutes)

Use `fixed_t2_fit` and `full_fit` to test whether freeing $T_2$ produces a statistically meaningful improvement.

Then inspect the fitted $T_2$ value and its covariance-based uncertainty from `full_fit_summary` to assess:

- the full model has a numerically lower $\chi^2$,
- the $\Delta\chi^2$ test does or does not show evidence for freeing $T_2$,
- the short observation window does or does not constrain $T_2$ precisely.


In [ ]:
# TODO 1: Calculate the Delta-chi-square comparison for fixed T2 vs free T2.

t2_test = pd.DataFrame([
    {
        "restricted_model": fixed_t2_fit["label"],
        "larger_model": full_fit["label"],
        "delta_chi2": ...,       # TODO
        "delta_parameters": ..., # TODO
        "p_value": ...,          # TODO
    }
])
t2_test


# Part 6: Residual comparison

A hypothesis-test table is not the end of the analysis. Always look at the data, fitted curves, and residuals. A global goodness-of-fit p-value can be acceptable even when the residuals show structure.


In [ ]:
plot_data = quantum_data.copy()
plot_smooth_data = smooth_quantum_data.copy()

models_to_plot = [fixed_t2_fit, full_fit]

for fit in models_to_plot:
    label = fit["label"]
    model_function = fit["model_function"]
    plot_data[label] = model_function(plot_data["time_us"], *fit["popt"])
    plot_data[f"{label} residual"] = (
        plot_data["probability"] - plot_data[label]
    ) / plot_data["sigma_probability"]
    plot_smooth_data[label] = model_function(
        plot_smooth_data["time_us"],
        *fit["popt"],
    )

fig, axes = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(8, 7),
    gridspec_kw={"height_ratios": [2, 1]},
)

axes[0].errorbar(
    plot_data["time_us"],
    plot_data["probability"],
    yerr=plot_data["sigma_probability"],
    fmt="o",
    capsize=3,
    color="black",
    label="data",
)

for fit in models_to_plot:
    axes[0].plot(
        plot_smooth_data["time_us"],
        plot_smooth_data[fit["label"]],
        label=fit["label"],
    )

axes[0].set_ylabel("probability")
axes[0].set_ylim(0.0, 1.0)
axes[0].legend(fontsize=9)

axes[1].axhline(0.0, color="0.5", lw=1)
for fit in models_to_plot:
    axes[1].plot(
        plot_data["time_us"],
        plot_data[f"{fit['label']} residual"],
        "o-",
        label=fit["label"],
    )

axes[1].set_xlabel("time (microseconds)")
axes[1].set_ylabel("norm. residual")
axes[1].legend(fontsize=9)
fig.tight_layout()


# Part 7: Takeaways

- **Weighted least squares** minimizes a sum of squared residuals measured in units of their uncertainties.
- **Goodness of fit** asks whether one model and its uncertainty assumptions produce residuals of a plausible overall size.
- Reduced $\chi^2\approx 1$ is a useful heuristic, but the Pearson p-value uses the full $\chi^2$ reference distribution with $\nu$ degrees of freedom.
- **Model comparison** is a different question: does relaxing a constraint improve the fit enough to justify extra flexibility?
- For suitable **nested** models, $\Delta\chi^2$ can be compared with $\chi^2_{\Delta k}$.
- The usual $\Delta\chi^2$ calibration can fail for weakly identified parameters, boundary hypotheses, strong nonlinearity, or poor optimization.
- A p-value is not the probability that a model is true.
- Confidence intervals should be described with coverage language: what the procedure would do over repeated experiments under the assumed model.
- Separate **statistical uncertainty** from **systematic uncertainty**; a covariance matrix usually covers only the uncertainty sources included in the measurement model.
- Numerical fit results should be interpreted together with residual plots and physical understanding.

**Bridge to Lecture 8:** when the simple asymptotic approximations are questionable, simulation/resampling and explicit likelihood-based methods provide more flexible ways to quantify uncertainty and test models.


# References and further reading

1. **D. W. Hogg, J. Bovy, and D. Lang**, *Data analysis recipes: Fitting a model to data*, arXiv:1008.4686.  
   https://arxiv.org/abs/1008.4686  
   Practical discussion of weighted least squares, residuals, uncertainty assumptions, and generative models. Also see the course [Unit 2 resources](../resources/unit-2-data-analysis.md).

2. **SciPy documentation: `scipy.optimize.curve_fit`**  
   https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html  
   Documents nonlinear least squares, the meaning of `sigma`, `absolute_sigma`, parameter covariance, and caveats for ill-conditioned fits.

3. **SciPy documentation: `scipy.stats.chi2`**  
   https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2.html  
   Reference for the $\chi^2$ distribution and the survival function `chi2.sf` used for upper-tail p-values.

4. **NIST/SEMATECH e-Handbook of Statistical Methods: Nonlinear Least Squares Regression**  
   https://www.itl.nist.gov/div898/handbook/pmd/section1/pmd142.htm  
   Overview of nonlinear least-squares models and practical conditions for fitting.

5. **S. S. Wilks**, “The Large-Sample Distribution of the Likelihood Ratio for Testing Composite Hypotheses,” *Annals of Mathematical Statistics* **9**, 60–62 (1938).  
   Classical asymptotic result underlying the $\Delta\chi^2$ reference distribution for regular nested hypotheses.

6. **G. Cowan, K. Cranmer, E. Gross, and O. Vitells**, “Asymptotic formulae for likelihood-based tests of new physics,” *Eur. Phys. J. C* **71**, 1554 (2011), arXiv:1007.1727.  
   https://arxiv.org/abs/1007.1727  
   More advanced discussion of asymptotic tests, including important regularity and boundary issues.


## Change log from the draft Lecture 7

- **Kept the Lecture 6 damped-oscillation model and short-time data window** so students can focus on statistics rather than learning a new physical system.
- **Reorganized the statistical sequence** to move from normalized residuals → $\chi^2$ → degrees of freedom → reduced $\chi^2$ → goodness-of-fit p-value → model comparison → nested hypotheses → $\Delta\chi^2$.
- **Added the conceptual origin of $\chi^2$** as a sum of squared standard-normal residuals and clarified why fitting parameters reduces the effective degrees of freedom.
- **Separated goodness of fit from model comparison explicitly** with a side-by-side table and discussion question.
- **Changed the main nested-model exercise to $\phi=0$ versus free $\phi$**, an interior hypothesis that gives a cleaner first example of $\Delta\chi^2$.
- **Retained fixed-$T_2$ versus free-$T_2$ as a second comparison** because the short time window naturally illustrates weak parameter identification.
- **Expanded caveats** on weak identification, optimizer bounds, overfitting, nonlinearity, and p-value interpretation.
- **Reduced likelihood emphasis** to a brief conceptual bridge; likelihood fitting remains reserved for Lecture 8.
- **Added three scaffolded coding activities totaling 32 minutes (40% of class)**. Exercise cells contain TODOs and partial structure without providing every solution.
- **Kept pandas DataFrames central** for data storage, fit summaries, and comparison tables.
- **Added reliable references** to Hogg et al., SciPy documentation, NIST, Wilks, and Cowan et al.
